# Métricas por dobra dos experimentos dos notebooks 02 e 03

O enunciado pede, no `experimentos.csv`, as métricas **por dobra**, a média e o desvio de cada experimento. Os notebooks 02 e 03 só guardaram a média e o desvio, então as linhas `H_*` do arquivo não têm as três dobras. Este notebook **refaz** esses experimentos com **exatamente as mesmas configurações** (mesmos modelos, mesmos hiperparâmetros, mesmo dataset, mesmas dobras, semente 42) e grava as métricas de cada dobra na mesma linha do `resultados/experimentos.csv`. Para conferir que é o mesmo experimento, ele imprime a **diferença entre a média nova e a anterior** (deve ser praticamente zero; só o paralelismo pode mexer na terceira casa decimal).

Só validação cruzada no desenvolvimento; o teste não é aberto. Os resultados já gravados são pulados, então o notebook pode ser interrompido e retomado.

| Grupo | O que refaz |
|---|---|
| 1 | Os 10 modelos em configuração inicial (`dataset_1_fundido`) e o baseline; KNN e stacking por último (são os mais lentos) |
| 2 | Os 3 melhores depois da busca (Gradient Boosting, bagging e Random Forest) |
| 3 | O Gradient Boosting nos 8 datasets |
| 4 | O treino sem as notas zero (4 modelos) |
| 5 | O PCA em 2 regressores (6 cortes cada) e o K-means como feature |

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import BaggingRegressor, HistGradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, PredefinedSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor

sys.path.insert(0, "../src")
from cv_utils import Registro, cv_dobras, protocolo

SMOKE = os.environ.get("SMOKE") == "1"
RES = Path(os.environ["SAIDA"]) if SMOKE else Path("../resultados")
PROC, ALVO = Path("../dados/processed"), "NU_NOTA_MT"
src_exp = Path("../resultados/experimentos.csv")
if SMOKE:
    pd.read_csv(src_exp).to_csv(RES / "experimentos.csv", index=False)           # cópia só para o teste do código
reg = Registro(RES / "experimentos.csv")
ids, em_dev, dobras = protocolo()
dobra_dev = pd.read_csv("../dados/splits/dobras_cv.csv")["dobra"].to_numpy()
amostra = np.sort(np.random.default_rng(42).choice(int(em_dev.sum()), 30_000, replace=False)) if SMOKE else None


def carregar(nome):
    dev = pd.read_csv(PROC / f"{nome}.csv")[em_dev].reset_index(drop=True)
    if SMOKE:
        dev = dev.iloc[amostra].reset_index(drop=True)
    return dev.drop(columns=ALVO), dev[ALVO]


if SMOKE:
    dobras = PredefinedSplit(dobra_dev[amostra])
X, y = carregar("dataset_1_fundido")
print(f"dataset_1_fundido: {len(X):,} pessoas no desenvolvimento, {X.shape[1]} features | {len(reg.exp)} linhas em experimentos.csv")


def hgb(**kw):
    base = dict(early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=42)
    return HistGradientBoostingRegressor(**{**base, **kw})


def gb_inicial():
    return hgb(learning_rate=0.1, max_iter=300, max_leaf_nodes=31, min_samples_leaf=200)


PARAMS_BUSCA = dict(l2_regularization=0.24400607090817547, learning_rate=0.06689135590392836, max_iter=500, max_leaf_nodes=89, min_samples_leaf=137)


def refazer(id_exp, modelo, Xc=None, yc=None, tempo_unico=False):
    """Refaz o experimento `id_exp` com as 3 dobras e atualiza a linha; imprime a diferença para a média anterior."""
    Xc, yc = (X, y) if Xc is None else (Xc, yc)
    if id_exp not in reg.exp.index:
        print(f"{id_exp}: não existe em experimentos.csv"); return
    if reg.tem_dobras(id_exp):
        print(f"{id_exp}: já tem as 3 dobras"); return
    antes = reg.exp.loc[id_exp, "RMSE_media"]
    d = cv_dobras(modelo, Xc, yc, dobras)
    nova = d["RMSE"].mean()
    obs = f" Refeito com as 3 dobras: RMSE médio {nova:.3f} (anterior {antes:.3f}, diferença {nova - antes:+.4f})."
    antigo = reg.exp.loc[id_exp, "conclusao"]
    reg.gravar(id_exp, d, conclusao=(antigo if isinstance(antigo, str) else "") + obs)
    print(f"{id_exp}: RMSE {nova:.3f} ± {d['RMSE'].std():.3f} (anterior {antes:.3f}; dif. {nova - antes:+.4f}) | por dobra {d['RMSE'].round(2).tolist()} | {d['tempo_s'].mean():.0f} s/dobra", flush=True)

dataset_1_fundido: 2,153,941 pessoas no desenvolvimento, 30 features | 82 linhas em experimentos.csv


## 1. Modelos em configuração inicial (notebook 02, `dataset_1_fundido`)

In [2]:
tree10 = lambda **kw: DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, **kw)
inicial = {
    "Baseline": DummyRegressor(strategy="mean"),
    "Regressão linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Árvore de decisão": tree10(random_state=42),
    "SVR linear": make_pipeline(StandardScaler(), LinearSVR(C=1.0, epsilon=0.0, loss="squared_epsilon_insensitive", dual=False, max_iter=5000, random_state=42)),
    "Bagging": BaggingRegressor(estimator=tree10(), n_estimators=20, bootstrap=True, n_jobs=-1, random_state=42),
    "Pasting": BaggingRegressor(estimator=tree10(), n_estimators=20, bootstrap=False, max_samples=0.5, n_jobs=-1, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=50, max_features=0.5, n_jobs=-1, random_state=42),
    "Gradient Boosting": gb_inicial(),
}
for nome, modelo in inicial.items():
    refazer(f"H_inicial_{nome}", modelo)

H_inicial_Baseline: RMSE 131.634 ± 0.158 (anterior 131.634; dif. +0.0001) | por dobra [131.52, 131.57, 131.81] | 0 s/dobra


H_inicial_Regressão linear: RMSE 111.153 ± 0.178 (anterior 111.153; dif. -0.0003) | por dobra [111.05, 111.05, 111.36] | 0 s/dobra


H_inicial_Ridge: RMSE 111.153 ± 0.178 (anterior 111.153; dif. -0.0003) | por dobra [111.05, 111.05, 111.36] | 0 s/dobra


H_inicial_Árvore de decisão: RMSE 110.551 ± 0.190 (anterior 110.551; dif. +0.0004) | por dobra [110.45, 110.44, 110.77] | 3 s/dobra


H_inicial_SVR linear: RMSE 111.153 ± 0.178 (anterior 111.153; dif. -0.0003) | por dobra [111.05, 111.05, 111.36] | 1 s/dobra


H_inicial_Bagging: RMSE 110.014 ± 0.207 (anterior 110.014; dif. +0.0001) | por dobra [109.92, 109.87, 110.25] | 14 s/dobra


H_inicial_Pasting: RMSE 110.012 ± 0.208 (anterior 110.012; dif. +0.0001) | por dobra [109.92, 109.87, 110.25] | 11 s/dobra


H_inicial_Random Forest: RMSE 109.422 ± 0.195 (anterior 109.422; dif. +0.0003) | por dobra [109.32, 109.3, 109.65] | 48 s/dobra


H_inicial_Gradient Boosting: RMSE 107.770 ± 0.226 (anterior 107.770; dif. +0.0005) | por dobra [107.71, 107.58, 108.02] | 17 s/dobra


## 2. Os três melhores depois da busca (notebook 02 otimizado)

In [3]:
refazer("H_busca_Gradient Boosting", hgb(**PARAMS_BUSCA))
refazer("H_busca_Bagging", BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=15, min_samples_leaf=34), n_estimators=24, max_samples=0.7, max_features=0.8,
                                            bootstrap=True, n_jobs=-1, random_state=42))
refazer("H_busca_Random Forest", RandomForestRegressor(n_estimators=50, max_depth=15, max_features=0.3, min_samples_leaf=30, n_jobs=-1, random_state=42))

H_busca_Gradient Boosting: RMSE 107.680 ± 0.218 (anterior 107.680; dif. +0.0000) | por dobra [107.65, 107.48, 107.91] | 29 s/dobra


H_busca_Bagging: RMSE 109.083 ± 0.189 (anterior 109.083; dif. +0.0000) | por dobra [108.98, 108.97, 109.3] | 19 s/dobra


H_busca_Random Forest: RMSE 109.138 ± 0.194 (anterior 109.138; dif. +0.0000) | por dobra [109.03, 109.02, 109.36] | 17 s/dobra


## 3. Gradient Boosting nos 8 datasets (notebook 02, comparação de datasets)

In [4]:
DATASETS = ["dataset_1_fundido", "dataset_1_features", "dataset_2_escola", "dataset_2_features_escola",
            "dataset_3_sem_escola", "dataset_3_sem_escola_features", "dataset_3_escola", "dataset_3_escola_features"]
for nome in DATASETS:
    Xd, yd = (X, y) if nome == "dataset_1_fundido" else carregar(nome)
    refazer(f"H_dataset_{nome}_GB", gb_inicial(), Xd, yd)

H_dataset_dataset_1_fundido_GB: RMSE 107.770 ± 0.226 (anterior 107.770; dif. +0.0005) | por dobra [107.71, 107.58, 108.02] | 18 s/dobra


H_dataset_dataset_1_features_GB: RMSE 107.759 ± 0.239 (anterior 107.759; dif. -0.0005) | por dobra [107.71, 107.55, 108.02] | 19 s/dobra


H_dataset_dataset_2_escola_GB: RMSE 107.261 ± 0.202 (anterior 107.261; dif. -0.0001) | por dobra [107.21, 107.09, 107.48] | 20 s/dobra


H_dataset_dataset_2_features_escola_GB: RMSE 107.255 ± 0.194 (anterior 107.255; dif. +0.0004) | por dobra [107.21, 107.09, 107.47] | 21 s/dobra


H_dataset_dataset_3_sem_escola_GB: RMSE 107.551 ± 0.211 (anterior 107.551; dif. -0.0004) | por dobra [107.49, 107.38, 107.79] | 20 s/dobra


H_dataset_dataset_3_sem_escola_features_GB: RMSE 107.540 ± 0.223 (anterior 107.540; dif. +0.0005) | por dobra [107.48, 107.35, 107.79] | 22 s/dobra


H_dataset_dataset_3_escola_GB: RMSE 107.056 ± 0.187 (anterior 107.056; dif. +0.0003) | por dobra [107.0, 106.91, 107.27] | 21 s/dobra


H_dataset_dataset_3_escola_features_GB: RMSE 107.048 ± 0.196 (anterior 107.048; dif. +0.0003) | por dobra [106.99, 106.89, 107.27] | 26 s/dobra


## 4. Treino sem as notas zero (notebook 02 sem zeros)

`SemZerosNoTreino` tira as pessoas com nota 0 **só do treino** de cada dobra; a validação mantém todos. Hiperparâmetros: os da busca daquele notebook.

In [5]:
class SemZerosNoTreino(RegressorMixin, BaseEstimator):
    def __init__(self, estimador, tirar_zeros=True):
        self.estimador = estimador
        self.tirar_zeros = tirar_zeros

    def fit(self, X, y):
        y = np.asarray(y)
        manter = (y > 0) if self.tirar_zeros else np.ones(len(y), dtype=bool)
        self.estimador_ = clone(self.estimador).fit(X.iloc[manter], y[manter])
        return self

    def predict(self, X):
        return self.estimador_.predict(X)


sem_zeros = {
    "Regressão linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=0.0029152036385288304)),
    "Árvore de decisão": DecisionTreeRegressor(max_depth=25, max_features=0.8, min_samples_leaf=298, random_state=42),
    "Gradient Boosting": hgb(**PARAMS_BUSCA),
}
for nome, est in sem_zeros.items():
    refazer(f"H_semzeros_{nome}", SemZerosNoTreino(est, tirar_zeros=True))

H_semzeros_Regressão linear: RMSE 111.202 ± 0.187 (anterior 111.202; dif. -0.0000) | por dobra [111.09, 111.09, 111.42] | 1 s/dobra


H_semzeros_Ridge: RMSE 111.202 ± 0.187 (anterior 111.202; dif. -0.0000) | por dobra [111.09, 111.09, 111.42] | 0 s/dobra


H_semzeros_Árvore de decisão: RMSE 110.331 ± 0.210 (anterior 110.331; dif. +0.0004) | por dobra [110.21, 110.21, 110.57] | 3 s/dobra


H_semzeros_Gradient Boosting: RMSE 107.737 ± 0.204 (anterior 107.737; dif. -0.0005) | por dobra [107.63, 107.61, 107.97] | 30 s/dobra


## 5. PCA em dois regressores e K-means como feature (notebook 03)

In [6]:
regs = {"Regressão linear": LinearRegression, "Gradient Boosting": gb_inicial}
for nome, fabrica in regs.items():
    refazer(f"H_pca_{nome}_sem PCA", make_pipeline(StandardScaler(), fabrica()))
    for k in (5, 10, 15, 20, 25):
        refazer(f"H_pca_{nome}_{k}", make_pipeline(StandardScaler(), PCA(n_components=k, random_state=42), fabrica()))
refazer("H_kmeans_Gradient Boosting", make_pipeline(StandardScaler(), make_union("passthrough", MiniBatchKMeans(n_clusters=2, n_init=3, batch_size=10_000, random_state=42)), gb_inicial()))

H_pca_Regressão linear_sem PCA: RMSE 111.153 ± 0.178 (anterior 111.153; dif. -0.0003) | por dobra [111.05, 111.05, 111.36] | 1 s/dobra


H_pca_Regressão linear_5: RMSE 114.425 ± 0.174 (anterior 114.425; dif. -0.0001) | por dobra [114.3, 114.36, 114.62] | 0 s/dobra


H_pca_Regressão linear_10: RMSE 113.047 ± 0.190 (anterior 113.047; dif. -0.0005) | por dobra [112.9, 112.98, 113.26] | 0 s/dobra


H_pca_Regressão linear_15: RMSE 112.718 ± 0.184 (anterior 112.718; dif. +0.0005) | por dobra [112.6, 112.63, 112.93] | 0 s/dobra


H_pca_Regressão linear_20: RMSE 111.887 ± 0.177 (anterior 111.887; dif. +0.0003) | por dobra [111.78, 111.79, 112.09] | 1 s/dobra


H_pca_Regressão linear_25: RMSE 111.658 ± 0.181 (anterior 111.658; dif. -0.0005) | por dobra [111.54, 111.56, 111.87] | 1 s/dobra


... (3 outputs left out)

H_pca_Gradient Boosting_15: RMSE 111.148 ± 0.181 (anterior 111.148; dif. +0.0003) | por dobra [111.03, 111.06, 111.36] | 9 s/dobra


H_pca_Gradient Boosting_20: RMSE 110.432 ± 0.172 (anterior 110.432; dif. +0.0004) | por dobra [110.36, 110.3, 110.63] | 9 s/dobra


H_pca_Gradient Boosting_25: RMSE 110.165 ± 0.183 (anterior 110.165; dif. -0.0003) | por dobra [110.05, 110.07, 110.38] | 10 s/dobra


H_kmeans_Gradient Boosting: RMSE 107.789 ± 0.232 (anterior 107.789; dif. -0.0000) | por dobra [107.74, 107.58, 108.04] | 22 s/dobra


## 6. KNN e stacking (os mais lentos)

KNN: cerca de 8 minutos por dobra. Stacking: cerca de 10 minutos por dobra. Ficam por último para o resto já estar gravado se o notebook for interrompido.

In [7]:
refazer("H_inicial_KNN", make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=50, algorithm="brute", n_jobs=-1)))
refazer("H_inicial_Stacking", StackingRegressor(
    estimators=[("ridge", make_pipeline(StandardScaler(), Ridge(alpha=1.0))), ("arvore", tree10(random_state=42)),
                ("gradient_boosting", hgb(learning_rate=0.1, max_iter=300, max_leaf_nodes=31, min_samples_leaf=200))],
    final_estimator=Ridge(alpha=1.0), cv=KFold(n_splits=3, shuffle=True, random_state=42)))

H_inicial_KNN: RMSE 111.575 ± 0.184 (anterior 111.575; dif. +0.0000) | por dobra [111.47, 111.47, 111.79] | 520 s/dobra


H_inicial_Stacking: RMSE 107.763 ± 0.226 (anterior 107.763; dif. -0.0000) | por dobra [107.71, 107.57, 108.01] | 63 s/dobra


## Conferência

Linhas `H_*` do `experimentos.csv` que agora têm as três dobras, com a diferença para a média anterior.

In [8]:
h = reg.exp[reg.exp.index.str.startswith("H_")]
print(f"{int(h['RMSE_d1'].notna().sum())} de {len(h)} linhas H_* com as 3 dobras")
h[["RMSE_media", "RMSE_desvio", "RMSE_d1", "RMSE_d2", "RMSE_d3"]].astype(float).round(3)

39 de 39 linhas H_* com as 3 dobras


,RMSE_media,RMSE_desvio,RMSE_d1,RMSE_d2,RMSE_d3
id,,,,,
H_inicial_Baseline,131.634,0.158,131.517,131.572,131.814
H_inicial_Regressão linear,111.153,0.178,111.050,111.050,111.358
H_inicial_Ridge,111.153,0.178,111.050,111.050,111.358
H_inicial_Árvore de decisão,110.551,0.190,110.446,110.438,110.771
H_inicial_KNN,111.575,0.184,111.468,111.469,111.788
H_inicial_SVR linear,111.153,0.178,111.050,111.050,111.358
H_inicial_Bagging,110.014,0.207,109.917,109.873,110.252
H_inicial_Pasting,110.012,0.208,109.918,109.868,110.251
H_inicial_Random Forest,109.422,0.195,109.319,109.301,109.647
